# Scraping WDumps data
<a href="https://colab.research.google.com/github/wmde/WDumps-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In order to better understand what kinds of entity data dump subsets our users are interested in, this repository scrapes all dump subsets listed under "recent dumps". The scrape includes a JSON representation of the filters that were used to generate the dump.

The notebook generates a csv file that includes filter data in a human-readable form. Each row of the csv includes the following columns:
- dump name
- URL
- filter (in human-readable form including labels for any items and properties used)
- statements included in the dump (in human-readable form)
- labels (yes/no)
- descriptions (yes/no)
- aliases (yes/no)
- sitelinks (yes/no)
- languages


## Preparing the workspace
### Installing dependencies

In [ ]:
%pip install wdumps_scraper

### Importing libraries

In [1]:
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
from enum import Enum
import json
import requests
import requests_cache
from operator import itemgetter
from wdumps_scraper import parsing

### Global definitions

In [ ]:
session = requests_cache.CachedSession(
    'scraper_cache',
    backend='sqlite',
    expire_after=None
)

class CacheDuration(Enum):
    NO_CACHE = 0
    LOW = 7200
    HIGH = None

def get(address, cache_duration: CacheDuration=CacheDuration.NO_CACHE ):
    response = session.get(address, expire_after=cache_duration.value)
    if response.status_code == 200:
      return response.text
    else:
      return False

def scrape_recent_dumps():
    url = "https://wdumps.toolforge.org/dumps"
    html = get(url)
    return parsing.extract_last_id(html)

def scrape_dump(dump_id):
    url = f"https://wdumps.toolforge.org/dump/{dump_id}"

    cache_duration = CacheDuration.HIGH if dump_id < last_id - 10 else CacheDuration.LOW

    html = get(url, cache_duration)
    if html:
        name = parsing.extract_name(html)
        filters = parsing.extract_filters(html)
        return True, {
            "url": url,
            "id": dump_id,
            "name": name,
            **filters
        }
    else:
        return False, dump_id


## Extract latest dump ID

In [ ]:
last_id = scrape_recent_dumps()
print(last_id)

## Extract IDs, names and inclusion information
Save these in a dictionary with the URLs.

In [ ]:
dumps = []
skipped = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {}

    for i in range(last_id, 0, -1):
        future = executor.submit(scrape_dump, i)
        futures[future] = i

    for future in as_completed(futures):
        id_ = futures[future]
        try:
            success, result = future.result()
            if success:
                dumps.append(result)
            else:
                skipped.append({"id": result, "error": "Could not retrieve URL."})
        except Exception as e:
            skipped.append({"id": id_, "error": str(e)})


print(len(dumps))
print(len(skipped))

## Save data to csv

In [ ]:
sorted_dumps = sorted(dumps, key=itemgetter("id"), reverse=True)

with open("wdumps-data.csv", "w", newline="") as csvfile:
  fieldnames = ["id", "name", "labels", "descriptions", "aliases", "sitelinks", "url"]
  writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
  writer.writeheader()

  for dump in sorted_dumps:
    writer.writerow(dump)